## SRP254063

**paper:** [PMID: 33188156](https://pmc.ncbi.nlm.nih.gov/articles/PMC11623972/) - The immune and metabolic changes with age in giant panda blood by combined transcriptome and DNA methylation analysis, 2020

**date, curator:** 2026-04-02, Sara Carsanaro

**notes**

### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,blood,UBERON:0000178,blood,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,28-year,UBERON:0007222,late adult stage,missing child term
2,12-year,UBERON:0018241,prime adult stage,missing child term
3,8-year,UBERON:0018241,prime adult stage,missing child term
4,7-year,UBERON:0018241,prime adult stage,missing child term
5,3-year,UBERON:0000112,sexually immature stage,missing child term
6,1-year,UBERON:0000112,sexually immature stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP254063"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 16/16 [00:15<00:00,  1.03it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX8004071,SRP254063,Illumina HiSeq 2500,SRS6378041,,,UBERON:0007222,late adult stage,,blood,28-year,,,missing child term,F,,,9646,,,,,,e2,SAMN14446256,28,year,,,,,,old,,,02/04/2026,28female,,e2,,,,old,,TRANSCRIPTOMIC,Oligo-dT
1,SRX8004070,SRP254063,Illumina HiSeq 2500,SRS6378040,,,UBERON:0007222,late adult stage,,blood,28-year,,,missing child term,M,,,9646,,,,,,e1,SAMN14446255,28,year,,,,,,old,,,02/04/2026,28male,,e1,,,,old,,TRANSCRIPTOMIC,Oligo-dT
2,SRX8004069,SRP254063,Illumina HiSeq 2500,SRS6378039,,,UBERON:0018241,prime adult stage,,blood,12-year,,,missing child term,F,,,9646,,,,,,d2,SAMN14446254,12,year,,,,,,adult,,,02/04/2026,12female,,d2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX8004068,SRP254063,Illumina HiSeq 2500,SRS6378038,,,UBERON:0018241,prime adult stage,,blood,8-year,,,missing child term,F,,,9646,,,,,,c2,SAMN14446253,8,year,,,,,,adult,,,02/04/2026,8female,,c2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX8004067,SRP254063,Illumina HiSeq 2500,SRS6378037,,,UBERON:0018241,prime adult stage,,blood,7-year,,,missing child term,M,,,9646,,,,,,c1,SAMN14446252,7,year,,,,,,adult,,,02/04/2026,7male,,c1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX8004066,SRP254063,Illumina HiSeq 2500,SRS6378036,,,UBERON:0000112,sexually immature stage,,blood,3-year,,,missing child term,M,,,9646,,,,,,b1,SAMN14446251,3,year,,,,,,young,,,02/04/2026,3male,,b1,,,,young,,TRANSCRIPTOMIC,Oligo-dT
6,SRX8004065,SRP254063,Illumina HiSeq 2500,SRS6378035,,,UBERON:0000112,sexually immature stage,,blood,1-year,,,missing child term,M,,,9646,,,,,,a2,SAMN14446250,1,year,,,,,,young,,,02/04/2026,1male,,a2,,,,young,,TRANSCRIPTOMIC,Oligo-dT
7,SRX8004064,SRP254063,Illumina HiSeq 2500,SRS6378034,,,UBERON:0000112,sexually immature stage,,blood,1-year,,,missing child term,F,,,9646,,,,,,a1,SAMN14446249,1,year,,,,,,young,,,02/04/2026,1female,,a1,,,,young,,TRANSCRIPTOMIC,Oligo-dT


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['blood']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000178'
library.loc[:,'anatName'] = 'blood'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX8004071,SRP254063,Illumina HiSeq 2500,SRS6378041,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,F,,,9646,,,,,,e2,SAMN14446256,28,year,,,,,,old,,,02/04/2026,28female,,e2,,,,old,,TRANSCRIPTOMIC,Oligo-dT
1,SRX8004070,SRP254063,Illumina HiSeq 2500,SRS6378040,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,M,,,9646,,,,,,e1,SAMN14446255,28,year,,,,,,old,,,02/04/2026,28male,,e1,,,,old,,TRANSCRIPTOMIC,Oligo-dT
2,SRX8004069,SRP254063,Illumina HiSeq 2500,SRS6378039,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,12-year,perfect match,not documented,missing child term,F,,,9646,,,,,,d2,SAMN14446254,12,year,,,,,,adult,,,02/04/2026,12female,,d2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX8004068,SRP254063,Illumina HiSeq 2500,SRS6378038,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,8-year,perfect match,not documented,missing child term,F,,,9646,,,,,,c2,SAMN14446253,8,year,,,,,,adult,,,02/04/2026,8female,,c2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX8004067,SRP254063,Illumina HiSeq 2500,SRS6378037,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,7-year,perfect match,not documented,missing child term,M,,,9646,,,,,,c1,SAMN14446252,7,year,,,,,,adult,,,02/04/2026,7male,,c1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX8004066,SRP254063,Illumina HiSeq 2500,SRS6378036,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,3-year,perfect match,not documented,missing child term,M,,,9646,,,,,,b1,SAMN14446251,3,year,,,,,,young,,,02/04/2026,3male,,b1,,,,young,,TRANSCRIPTOMIC,Oligo-dT
6,SRX8004065,SRP254063,Illumina HiSeq 2500,SRS6378035,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,1-year,perfect match,not documented,missing child term,M,,,9646,,,,,,a2,SAMN14446250,1,year,,,,,,young,,,02/04/2026,1male,,a2,,,,young,,TRANSCRIPTOMIC,Oligo-dT
7,SRX8004064,SRP254063,Illumina HiSeq 2500,SRS6378034,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,1-year,perfect match,not documented,missing child term,F,,,9646,,,,,,a1,SAMN14446249,1,year,,,,,,young,,,02/04/2026,1female,,a1,,,,young,,TRANSCRIPTOMIC,Oligo-dT


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['1-year' '12-year' '28-year' '3-year' '7-year' '8-year']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX8004071,SRP254063,Illumina HiSeq 2500,SRS6378041,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,e2,SAMN14446256,28,year,,,,,,old,,,02/04/2026,28female,,e2,,,,old,,TRANSCRIPTOMIC,Oligo-dT
1,SRX8004070,SRP254063,Illumina HiSeq 2500,SRS6378040,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,e1,SAMN14446255,28,year,,,,,,old,,,02/04/2026,28male,,e1,,,,old,,TRANSCRIPTOMIC,Oligo-dT
2,SRX8004069,SRP254063,Illumina HiSeq 2500,SRS6378039,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,12-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,d2,SAMN14446254,12,year,,,,,,adult,,,02/04/2026,12female,,d2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX8004068,SRP254063,Illumina HiSeq 2500,SRS6378038,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,8-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,c2,SAMN14446253,8,year,,,,,,adult,,,02/04/2026,8female,,c2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX8004067,SRP254063,Illumina HiSeq 2500,SRS6378037,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,7-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,c1,SAMN14446252,7,year,,,,,,adult,,,02/04/2026,7male,,c1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX8004066,SRP254063,Illumina HiSeq 2500,SRS6378036,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,3-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,b1,SAMN14446251,3,year,,,,,,young,,,02/04/2026,3male,,b1,,,,young,,TRANSCRIPTOMIC,Oligo-dT
6,SRX8004065,SRP254063,Illumina HiSeq 2500,SRS6378035,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,1-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,a2,SAMN14446250,1,year,,,,,,young,,,02/04/2026,1male,,a2,,,,young,,TRANSCRIPTOMIC,Oligo-dT
7,SRX8004064,SRP254063,Illumina HiSeq 2500,SRS6378034,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,1-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,a1,SAMN14446249,1,year,,,,,,young,,,02/04/2026,1female,,a1,,,,young,,TRANSCRIPTOMIC,Oligo-dT


#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-07'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX8004071,SRP254063,Illumina HiSeq 2500,SRS6378041,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,e2,SAMN14446256,28,year,,,,,,old,,SAC,2026-04-07,28female,,e2,,,,old,,TRANSCRIPTOMIC,Oligo-dT
1,SRX8004070,SRP254063,Illumina HiSeq 2500,SRS6378040,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,e1,SAMN14446255,28,year,,,,,,old,,SAC,2026-04-07,28male,,e1,,,,old,,TRANSCRIPTOMIC,Oligo-dT
2,SRX8004069,SRP254063,Illumina HiSeq 2500,SRS6378039,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,12-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,d2,SAMN14446254,12,year,,,,,,adult,,SAC,2026-04-07,12female,,d2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX8004068,SRP254063,Illumina HiSeq 2500,SRS6378038,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,8-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,c2,SAMN14446253,8,year,,,,,,adult,,SAC,2026-04-07,8female,,c2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX8004067,SRP254063,Illumina HiSeq 2500,SRS6378037,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,7-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,c1,SAMN14446252,7,year,,,,,,adult,,SAC,2026-04-07,7male,,c1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX8004066,SRP254063,Illumina HiSeq 2500,SRS6378036,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,3-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,b1,SAMN14446251,3,year,,,,,,young,,SAC,2026-04-07,3male,,b1,,,,young,,TRANSCRIPTOMIC,Oligo-dT
6,SRX8004065,SRP254063,Illumina HiSeq 2500,SRS6378035,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,1-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,a2,SAMN14446250,1,year,,,,,,young,,SAC,2026-04-07,1male,,a2,,,,young,,TRANSCRIPTOMIC,Oligo-dT
7,SRX8004064,SRP254063,Illumina HiSeq 2500,SRS6378034,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,1-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,a1,SAMN14446249,1,year,,,,,,young,,SAC,2026-04-07,1female,,a1,,,,young,,TRANSCRIPTOMIC,Oligo-dT


#### comments

In [11]:
library.loc[:,'comment'] = 'PMID: 33188156'

#### save complete file with correct columns

In [12]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX8004071,SRP254063,Illumina HiSeq 2500,SRS6378041,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,e2,SAMN14446256,28,year,,,,,PMID: 33188156,old,,SAC,2026-04-07
1,SRX8004070,SRP254063,Illumina HiSeq 2500,SRS6378040,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,e1,SAMN14446255,28,year,,,,,PMID: 33188156,old,,SAC,2026-04-07
2,SRX8004069,SRP254063,Illumina HiSeq 2500,SRS6378039,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,12-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,d2,SAMN14446254,12,year,,,,,PMID: 33188156,adult,,SAC,2026-04-07
3,SRX8004068,SRP254063,Illumina HiSeq 2500,SRS6378038,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,8-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,c2,SAMN14446253,8,year,,,,,PMID: 33188156,adult,,SAC,2026-04-07
4,SRX8004067,SRP254063,Illumina HiSeq 2500,SRS6378037,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,7-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,c1,SAMN14446252,7,year,,,,,PMID: 33188156,adult,,SAC,2026-04-07
5,SRX8004066,SRP254063,Illumina HiSeq 2500,SRS6378036,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,3-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,b1,SAMN14446251,3,year,,,,,PMID: 33188156,young,,SAC,2026-04-07
6,SRX8004065,SRP254063,Illumina HiSeq 2500,SRS6378035,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,1-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,a2,SAMN14446250,1,year,,,,,PMID: 33188156,young,,SAC,2026-04-07
7,SRX8004064,SRP254063,Illumina HiSeq 2500,SRS6378034,UBERON:0000178,blood,UBERON:0000112,sexually immature stage,,blood,1-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,a1,SAMN14446249,1,year,,,,,PMID: 33188156,young,,SAC,2026-04-07


### experiment annotations

In [13]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP254063,Ailuropoda melanoleuca Raw sequence reads,DNA methylation and transcriptome analysis revealed the molecular mechanism of physiological changes with age in giant pandas,SRA,,,,,,,PRJNA615061,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [14]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

8

In [15]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP254063,Ailuropoda melanoleuca Raw sequence reads,DNA methylation and transcriptome analysis revealed the molecular mechanism of physiological changes with age in giant pandas,SRA,partial,Bgee 1K,8,,,,PRJNA615061,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [16]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '33188156'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11623972/'
experiment.loc[:,'DOI'] = '10.18632/aging.103990'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP254063,Ailuropoda melanoleuca Raw sequence reads,DNA methylation and transcriptome analysis revealed the molecular mechanism of physiological changes with age in giant pandas,SRA,partial,Bgee 1K,8,,,,PRJNA615061,33188156,https://pmc.ncbi.nlm.nih.gov/articles/PMC11623972/,10.18632/aging.103990,,


#### comments

In [17]:
experiment.loc[:,'comment'] = 'removed methylation libraries'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP254063,Ailuropoda melanoleuca Raw sequence reads,DNA methylation and transcriptome analysis revealed the molecular mechanism of physiological changes with age in giant pandas,SRA,partial,Bgee 1K,8,,,,PRJNA615061,33188156,https://pmc.ncbi.nlm.nih.gov/articles/PMC11623972/,10.18632/aging.103990,,removed methylation libraries


#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [20]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
58851,SRX6439032,SRP214540,Illumina HiSeq 2000,SRS5093626,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,11 years,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,P1,SAMN12268509,11,year,,,,,PMID: 32344048,Late pregnancy,Late pregnancy,SAC,2026-04-02
58852,SRX6439031,SRP214540,Illumina HiSeq 2000,SRS5093625,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,18 years,perfect match,not documented,other,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,P2,SAMN12268510,18,year,,,,,PMID: 32344048,Late pregnancy,Late pregnancy,SAC,2026-04-02
58853,SRX8004071,SRP254063,Illumina HiSeq 2500,SRS6378041,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,e2,SAMN14446256,28,year,,,,,PMID: 33188156,old,,SAC,2026-04-07
58854,SRX8004070,SRP254063,Illumina HiSeq 2500,SRS6378040,UBERON:0000178,blood,UBERON:0007222,late adult stage,,blood,28-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,e1,SAMN14446255,28,year,,,,,PMID: 33188156,old,,SAC,2026-04-07
58855,SRX8004069,SRP254063,Illumina HiSeq 2500,SRS6378039,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,12-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,d2,SAMN14446254,12,year,,,,,PMID: 33188156,adult,,SAC,2026-04-07
58856,SRX8004068,SRP254063,Illumina HiSeq 2500,SRS6378038,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,8-year,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,c2,SAMN14446253,8,year,,,,,PMID: 33188156,adult,,SAC,2026-04-07
58857,SRX8004067,SRP254063,Illumina HiSeq 2500,SRS6378037,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,blood,7-year,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,c1,SAMN14446252,7,year,,,,,PMID: 33188156,adult,,SAC,2026-04-07


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1146,SRP174344,Blood transcriptome reveal immune response of ...,20 blood transcriptome data of giant panda wer...,SRA,partial,Bgee 1K,5,,,,PRJNA511547,31473266,https://www.sciencedirect.com/science/article/...,10.1016/j.dci.2019.103489,,"removed samples that have been vaccinated, alt..."
1147,SRP214540,giant panda blood transcriptomes,The giant panda RNA-seq data for different pha...,SRA,total,Bgee 1K,20,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA554475,32344048,https://www.sciencedirect.com/science/article/...,10.1016/j.dci.2020.103699,,
1148,SRP254063,Ailuropoda melanoleuca Raw sequence reads,DNA methylation and transcriptome analysis rev...,SRA,partial,Bgee 1K,8,,,,PRJNA615061,33188156,https://pmc.ncbi.nlm.nih.gov/articles/PMC11623...,10.18632/aging.103990,,removed methylation libraries


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [28]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [29]:
! git add $git_experiment_path $git_library_path

In [30]:
! git commit -m $commit_message_exp

[develop 9b42ae6] adding annotated bulk experiment SRP254063
 2 files changed, 9 insertions(+)


In [31]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.04 KiB | 2.04 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   ab9a7db..9b42ae6  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push